In [1]:
import sys
import os
# 将项目根目录加入路径，以便识别 01_Environment 模块
sys.path.append(os.path.abspath('..'))

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

from 01_Environment.liquidation_env import LiquidationEnv

# 初始化环境
env = LiquidationEnv()

# 使用 SB3 自带工具检查环境是否符合规范 (如果不报错就说明环境写得很完美)
check_env(env)
print("环境检查通过！")


SyntaxError: invalid decimal literal (2399528802.py, line 11)

In [ ]:
# 使用 Monitor 包装环境以记录训练数据
log_dir = "../models/logs/"
os.makedirs(log_dir, exist_ok=True)
env = Monitor(LiquidationEnv(), log_dir)

# 评估环境与 Callback (边训练边测试)
eval_env = Monitor(LiquidationEnv(), log_dir)
eval_callback = EvalCallback(eval_env, best_model_save_path='../models/best_ppo_model/',
                             log_path=log_dir, eval_freq=2000,
                             deterministic=True, render=False)

# 初始化 PPO 模型
# MlpPolicy 适用于一维向量状态
model = PPO("MlpPolicy", env, verbose=1, 
            learning_rate=3e-4, 
            n_steps=2048, 
            batch_size=64, 
            gamma=1.0) # gamma=1.0 因为我们的 reward 是真正的资金成本，不需要时间折扣

# 开始训练 (由于是金融时序，建议步数设大一点，比如 100k 到 500k)
print("开始训练...")
model.learn(total_timesteps=100000, callback=eval_callback)

# 保存最终模型
model.save("../models/ppo_liquidation_final")
print("训练完成并保存模型！")
